# 🛣️ ROADGUARD AI — HUẤN LUYỆN YOLOv11 CHUYÊN DỤNG VẾT NỨT MẶT ĐƯỜNG (ROAD CRACK)
### Dataset: `Road Crack.v1-dataset.yolov11.zip` (~530 MB | 6,315 ảnh 640x640)
---
> **Mục tiêu:** Huấn luyện mô hình YOLOv11 chuyên sâu nhận diện vết nứt mặt đường (dọc, ngang, lưới chân chim, nứt mép) phục vụ hệ thống UAV RoadGuard.
> **Bộ dữ liệu:** Roboflow Universe (`road-crack-7jw6e` v1 - khảo sát thực tế đường nhựa & bê tông).
> **Phân bổ:**
> - Tập Train: **5,531 ảnh** + 5,531 nhãn `.txt`
> - Tập Validation: **519 ảnh** + 519 nhãn `.txt`
> - Tập Test độc lập: **265 ảnh** + 265 nhãn `.txt`
> **Class nhận diện:** `crack` (0)
> **Phần cứng khuyến nghị:** Google Colab GPU Tesla T4 (Miễn phí) hoặc GPU cá nhân (RTX series).
> **Thời gian dự kiến:** ~35–50 phút / 50 epochs (batch=16, imgsz=640) trên T4 GPU.

## 📌 Bước 1: Kiểm Tra GPU & Cài Đặt Môi Trường Ultralytics YOLOv11

In [ ]:
# 1. Kiểm tra thông tin GPU
!nvidia-smi

import sys, os
import torch
print('=' * 65)
print(f'CUDA Available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU Model:      {torch.cuda.get_device_name(0)}')
    print(f'VRAM Capacity:  {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')
else:
    print('⚠️ CẢNH BÁO: Chưa kích hoạt GPU! Vào Runtime -> Change runtime type -> Chọn T4 GPU.')
print('=' * 65)

# 2. Cài đặt thư viện Ultralytics YOLOv11 & dependencies
!pip install -q ultralytics albumentations opencv-python matplotlib pyyaml

## 📥 Bước 2: Nạp & Giải Nén Dataset `Road Crack.v1-dataset.yolov11.zip`
---
> **Hướng dẫn nạp dữ liệu (Chọn 1 trong các cách sau):**
> 
> * **Cách 1 (Google Drive - Tốc độ cao nhất, khuyên dùng):** Tải file `Road Crack.v1-dataset.yolov11.zip` lên Google Drive của bạn, sau đó bỏ dấu `#` ở ô Mount Drive bên dưới để sao chép chỉ mất vài giây.
> * **Cách 2 (Kéo thả vào Colab):** Bấm biểu tượng 📁 **Files** ở thanh công cụ bên trái Colab, kéo thả file `Road Crack.v1-dataset.yolov11.zip` vào, đợi vòng tròn tải lên hoàn tất 100%.
> * **Cách 3 (Chạy Local trên máy tính):** Nếu bạn mở notebook này trực tiếp bằng Jupyter Notebook/VSCode trên máy cá nhân, code sẽ tự động đọc trực tiếp từ đường dẫn `F:\Fake_admin\Downloads\tổng hợp thông tin\Road Crack.v1-dataset.yolov11.zip`.

In [ ]:
# (Tùy chọn) Bỏ dấu '#' ở 2 dòng dưới nếu bạn lưu file zip trên Google Drive
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
import os, glob, shutil, zipfile
from pathlib import Path
import yaml

# Thư mục đích giải nén
dataset_dir = Path('/content/dataset') if os.path.exists('/content') else Path('./dataset_road_crack')
dataset_dir.mkdir(parents=True, exist_ok=True)

# Tìm file zip theo các nguồn khả thi
local_pc_path = Path(r'F:\Fake_admin\Downloads\tổng hợp thông tin\Road Crack.v1-dataset.yolov11.zip')
candidates = [
    local_pc_path,
    Path('/content/Road Crack.v1-dataset.yolov11.zip'),
    Path('/content/road_crack.zip'),
    Path('/content/drive/MyDrive/Road Crack.v1-dataset.yolov11.zip'),
    Path('/content/drive/MyDrive/road_crack.zip'),
]
# Quét thêm tất cả các file zip hiện có trong thư mục làm việc
for z in glob.glob('/content/*.zip') + glob.glob('*.zip') + glob.glob('/content/drive/MyDrive/*.zip'):
    candidates.append(Path(z))

chosen_zip = None
for c in candidates:
    if c.exists() and c.is_file() and zipfile.is_zipfile(c):
        chosen_zip = c
        break

if not chosen_zip:
    raise FileNotFoundError(
        "❌ CHƯA TÌM THẤY FILE ZIP DATASET!\n"
        "👉 Hãy kéo thả file 'Road Crack.v1-dataset.yolov11.zip' vào cột Files (📁) bên trái Colab\n"
        "hoặc tải lên Google Drive rồi Mount Drive."
    )

size_mb = chosen_zip.stat().st_size / (1024 * 1024)
print(f'✅ Đã tìm thấy dataset zip: {chosen_zip}')
print(f'📊 Dung lượng file: {size_mb:.2f} MB')

# Giải nén
print(f'📦 Đang giải nén vào {dataset_dir}...')
with zipfile.ZipFile(chosen_zip, 'r') as zf:
    zf.extractall(dataset_dir)
print('🎉 Giải nén hoàn tất!')

## 🔧 Bước 3: Chuẩn Hóa Đường Dẫn `data.yaml` & Kiểm Tra Số Lượng Ảnh
---
> ⚠️ **Lưu ý kỹ thuật:** File `data.yaml` xuất từ Roboflow thường dùng đường dẫn tương đối `../train/images`. Ô mã bên dưới sẽ tự động chuyển thành cấu hình tuyệt đối chuẩn xác 100% để YOLOv11 không bao giờ bị lỗi path.

In [ ]:
# 1. Tìm và chuẩn hóa data.yaml
yaml_files = list(dataset_dir.glob('**/data.yaml')) + list(dataset_dir.glob('data.yaml'))
if not yaml_files:
    raise FileNotFoundError('Không tìm thấy file data.yaml trong thư mục giải nén!')

yaml_path = yaml_files[0]
print(f'📄 Định vị file cấu hình gốc: {yaml_path}')

with open(yaml_path, 'r', encoding='utf-8') as f:
    yaml_data = yaml.safe_load(f)

# Chuẩn hóa path tuyệt đối
train_img_dir = dataset_dir / 'train' / 'images'
valid_img_dir = dataset_dir / 'valid' / 'images'
test_img_dir  = dataset_dir / 'test' / 'images'

# Nếu roboflow nằm trong subfolder
if not train_img_dir.exists():
    for sub in dataset_dir.glob('**/train/images'):
        train_img_dir = sub
        valid_img_dir = sub.parent.parent / 'valid' / 'images'
        test_img_dir  = sub.parent.parent / 'test' / 'images'
        break

dataset_root = train_img_dir.parent.parent
normalized_cfg = {
    'path': str(dataset_root.resolve()).replace('\\', '/'),
    'train': 'train/images',
    'val': 'valid/images',
    'test': 'test/images' if test_img_dir.exists() else 'valid/images',
    'nc': 1,
    'names': ['crack']
}

clean_yaml_path = dataset_dir / 'road_crack_normalized.yaml'
with open(clean_yaml_path, 'w', encoding='utf-8') as f:
    yaml.dump(normalized_cfg, f, default_flow_style=False)

print('=' * 65)
print('⚙️ CẤU HÌNH DATASET ĐÃ ĐƯỢC CHUẨN HÓA:')
with open(clean_yaml_path, 'r', encoding='utf-8') as f:
    print(f.read())
print('=' * 65)

# 2. Thống kê số lượng ảnh và file nhãn .txt
def count_samples(img_dir):
    if not img_dir.exists(): return 0, 0
    imgs = [f for f in img_dir.glob('*.*') if f.suffix.lower() in ['.jpg', '.jpeg', '.png', '.webp', '.bmp']]
    lbl_dir = img_dir.parent / 'labels'
    lbls = list(lbl_dir.glob('*.txt')) if lbl_dir.exists() else []
    return len(imgs), len(lbls)

n_train_img, n_train_lbl = count_samples(train_img_dir)
n_val_img, n_val_lbl     = count_samples(valid_img_dir)
n_test_img, n_test_lbl   = count_samples(test_img_dir)

print('📊 THỐNG KÊ DATASET THỰC TẾ:')
print(f' - Tập Train:      {n_train_img:,} ảnh  |  {n_train_lbl:,} file nhãn .txt')
print(f' - Tập Validation: {n_val_img:,} ảnh    |  {n_val_lbl:,} file nhãn .txt')
print(f' - Tập Test:       {n_test_img:,} ảnh   |  {n_test_lbl:,} file nhãn .txt')
print(f' 👉 Tổng cộng:     {n_train_img + n_val_img + n_test_img:,} ảnh sẵn sàng huấn luyện!')
print('=' * 65)

## 🖼️ Bước 4: Trực Quan Hóa Mẫu Dữ Liệu & Hộp Giới Hạn Bounding Box
---

In [ ]:
import cv2
import random
import matplotlib.pyplot as plt

def preview_dataset_samples(img_dir, num_samples=6):
    img_files = [f for f in img_dir.glob('*.*') if f.suffix.lower() in ['.jpg', '.jpeg', '.png']]
    if not img_files:
        print('Không có ảnh trong thư mục')
        return
    
    samples = random.sample(img_files, min(num_samples, len(img_files)))
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes = axes.flatten()
    
    for idx, img_path in enumerate(samples):
        img = cv2.imread(str(img_path))
        if img is None: continue
        h, w = img.shape[:2]
        
        lbl_path = img_path.parent.parent / 'labels' / f'{img_path.stem}.txt'
        box_count = 0
        if lbl_path.exists():
            with open(lbl_path, 'r') as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) >= 5:
                        cls_id = int(parts[0])
                        x_c, y_c, bw, bh = map(float, parts[1:5])
                        x1 = int((x_c - bw / 2) * w)
                        y1 = int((y_c - bh / 2) * h)
                        x2 = int((x_c + bw / 2) * w)
                        y2 = int((y_c + bh / 2) * h)
                        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 0, 255), 2)
                        cv2.putText(img, 'crack', (x1, max(y1 - 6, 15)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2)
                        box_count += 1
                        
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        axes[idx].imshow(img_rgb)
        axes[idx].set_title(f'{img_path.name[:20]}... ({box_count} cracks)')
        axes[idx].axis('off')
        
    plt.tight_layout()
    plt.show()

print('🔍 Đang tải xem trước 6 mẫu ảnh trong tập Train...')
preview_dataset_samples(train_img_dir, num_samples=6)

## 🚀 Bước 5: Huấn Luyện Mô Hình YOLOv11 Chuyên Sâu Vết Nứt Mặt Đường
---
> **Cấu hình tối ưu:**
> - Mô hình gốc: `yolo11n.pt` (hoặc bạn có thể đổi thành `yolo11s.pt` nếu muốn mAP cao hơn).
> - `epochs=50`, `patience=15` (tự dừng khi đạt đỉnh tối ưu).
> - `imgsz=640`, `batch=16` (an toàn trên T4 GPU 15GB VRAM).

In [ ]:
from ultralytics import YOLO

# 1. Tải kiến trúc nền tảng YOLOv11 Nano (pretrained COCO)
model = YOLO('yolo11n.pt')

# 2. Cấu hình & thực thi huấn luyện
run_name = 'roadguard_crack_yolo11n_run'
print('=' * 65)
print('🚀 BẮT ĐẦU HUẤN LUYỆN YOLOV11 NANO TRÊN DATASET ROAD CRACK...')
print(f'🎯 File cấu hình: {clean_yaml_path}')
print('=' * 65)

train_results = model.train(
    data=str(clean_yaml_path),
    epochs=50,
    patience=15,             # Dừng sớm nếu không cải thiện sau 15 epochs
    imgsz=640,
    batch=16,
    device=0 if torch.cuda.is_available() else 'cpu',
    workers=4,
    optimizer='auto',        # Tự động chọn SGD hoặc AdamW tối ưu
    lr0=0.01,
    weight_decay=0.0005,
    mosaic=1.0,              # Tăng cường dữ liệu Mosaic
    degrees=10.0,            # Xoay nhẹ góc camera flycam
    fliplr=0.5,              # Lật ngang
    name=run_name,
    project='runs/detect',
    save=True,
    save_period=10,
    plots=True
)

print('=' * 65)
print('🎉 TIẾN TRÌNH HUẤN LUYỆN ĐÃ HOÀN TẤT THÀNH CÔNG!')
print('=' * 65)

## 📊 Bước 6: Trực Quan Hóa Đồ Thị Hội Tụ & Ma Trận Nhầm Lẫn (Validation Metrics)

In [ ]:
from IPython.display import Image, display

run_dir = Path('runs/detect') / run_name
print(f'📁 Thư mục kết quả: {run_dir.resolve()}')

# 1. Đồ thị Loss & Metrics
res_img = run_dir / 'results.png'
if res_img.exists():
    print('📈 Biểu đồ Hàm Mất Mát (Losses) & Độ Chính Xác (mAP@50, mAP@50-95):')
    display(Image(str(res_img)))

# 2. Ma trận nhầm lẫn
cm_img = run_dir / 'confusion_matrix.png'
if cm_img.exists():
    print('🎯 Ma Trận Nhầm Lẫn (Confusion Matrix):')
    display(Image(str(cm_img)))

# 3. Đường cong F1-Score
f1_img = run_dir / 'F1_curve.png'
if f1_img.exists():
    print('⚖️ Đường cong F1-Score theo ngưỡng tin cậy:')
    display(Image(str(f1_img)))

# 4. Ảnh dự đoán trực tiếp trên tập Validation
val_batch_preds = list(run_dir.glob('val_batch*_pred.jpg'))
if val_batch_preds:
    print(f'🔍 Dự đoán mẫu trên tập Validation ({val_batch_preds[0].name}):')
    display(Image(str(val_batch_preds[0])))

## 🧪 Bước 7: Kiểm Thử Độc Lập Trên Tập Test (265 Ảnh Chưa Từng Thấy)

In [ ]:
# 1. Nạp file trọng số tốt nhất vừa huấn luyện
best_model_path = run_dir / 'weights' / 'best.pt'
if not best_model_path.exists():
    raise FileNotFoundError(f'Không tìm thấy file {best_model_path}')

trained_model = YOLO(str(best_model_path))

# 2. Đánh giá trên tập Test độc lập
print('🧪 Đang đánh giá model trên tập Test (265 ảnh)...\n')
test_metrics = trained_model.val(data=str(clean_yaml_path), split='test')

print('=' * 65)
print('🏆 KẾT QUẢ ĐÁNH GIÁ TRÊN TẬP TEST ĐỘC LẬP:')
print(f' - Precision (Độ chuẩn xác):     {test_metrics.box.mp * 100:.2f}%')
print(f' - Recall (Độ bao quát):         {test_metrics.box.mr * 100:.2f}%')
print(f' - mAP@50 (Độ chuẩn ở IoU 0.5):   {test_metrics.box.map50 * 100:.2f}%')
print(f' - mAP@50-95 (Tổng hợp đa IoU):  {test_metrics.box.map * 100:.2f}%')
print('=' * 65)

# 3. Dự đoán thử nghiệm 4 ảnh Test ngẫu nhiên
test_img_files = list(test_img_dir.glob('*.*'))
if test_img_files:
    test_samples = random.sample(test_img_files, min(4, len(test_img_files)))
    pred_res = trained_model.predict(source=[str(s) for s in test_samples], conf=0.25, save=False)
    
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    for i, r in enumerate(pred_res):
        plot_img = r.plot()
        plot_rgb = cv2.cvtColor(plot_img, cv2.COLOR_BGR2RGB)
        axes[i].imshow(plot_rgb)
        axes[i].set_title(f'Test {i+1}: {len(r.boxes)} vết nứt')
        axes[i].axis('off')
    plt.tight_layout()
    plt.show()

## 💾 Bước 8: Đóng Gói Trọng Số & Tải Về Máy Tính / Sao Lưu Drive
---
> File trọng số được đổi tên thành `roadguard_crack_yolo11n_best.pt`.
> Sau khi tải về, bạn chỉ việc đưa file này vào thư mục dự án:
> 👉 `f:\train AI đồ án\weights\trained\roadguard_crack_yolo11n_best.pt`
> Giao diện Web Test Bench và Worker API sẽ lập tức tự động nhận diện mô hình mới này trong danh sách!

In [ ]:
import hashlib

output_weight_name = 'roadguard_crack_yolo11n_best.pt'
output_weight_path = Path(output_weight_name)
shutil.copy(best_model_path, output_weight_path)

# Tính kích thước và mã băm SHA256
size_mb = output_weight_path.stat().st_size / (1024 * 1024)
hasher = hashlib.sha256()
with open(output_weight_path, 'rb') as f:
    hasher.update(f.read())
sha256_hash = hasher.hexdigest()

print('=' * 65)
print('✅ ĐÃ SẢN XUẤT FILE TRỌNG SỐ HOÀN CHỈNH:')
print(f' - Tên file:     {output_weight_name}')
print(f' - Dung lượng:   {size_mb:.2f} MB')
print(f' - SHA256:       {sha256_hash[:16]}...{sha256_hash[-8:]}')
print('=' * 65)

# Tự động sao lưu sang Google Drive nếu đang mount
drive_dest = Path('/content/drive/MyDrive/RoadGuard_Models')
if drive_dest.parent.exists():
    drive_dest.mkdir(parents=True, exist_ok=True)
    shutil.copy(output_weight_path, drive_dest / output_weight_name)
    print(f'☁️ Đã sao lưu an toàn vào Google Drive: {drive_dest / output_weight_name}')

# Tải về máy tính qua trình duyệt (Colab)
try:
    from google.colab import files
    print('⬇️ Đang kích hoạt hộp thoại tải file về máy tính qua trình duyệt...')
    files.download(str(output_weight_path))
except Exception as e:
    print(f'ℹ️ (Môi trường local hoặc không có files.download): File đã lưu tại {output_weight_path.resolve()}')

print('\n👉 Bước tiếp theo: Sao chép file trọng số này vào thư mục:')
print(r'   F:\train AI đồ án\weights\trained\roadguard_crack_yolo11n_best.pt')
print('   Sau đó mở http://127.0.0.1:8080 để kiểm thử trực quan trên Web UI!')